In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys
import time

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive as colab_drive
    colab_drive.mount("/content/drive", force_remount=False)
    WORKSPACE = Path("/content/drive/MyDrive/Zhong et al. 2025 - Neuromatch Team Workspace")
    CODE = WORKSPACE / "code"
    CACHE = Path("/content/zhong-cache")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scipy>=1.11,<2"], check=True)
    DATABASE = WORKSPACE / "zhong.duckdb"
else:
    WORKSPACE = next(
        candidate for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "code").is_dir()
        and (candidate / "data" / "cache" / "zhong.duckdb").is_file()
    )
    CODE = WORKSPACE / "code"
    CACHE = WORKSPACE / "data" / "cache"
    DATABASE = CACHE / "zhong.duckdb"
os.environ.setdefault("MPLCONFIGDIR", str(WORKSPACE / ".matplotlib"))
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))

import matplotlib.pyplot as plt
import pandas as pd

import drive

db = drive.setup(cache=str(CACHE), database=str(DATABASE), mount=False)
assert db.database_path.is_file()
db


In [ ]:
from dprime import AREAS, dprime_summary_tables, session_dprime_history

WINDOW_PAIRS = 20
NEURONS_PER_AREA = 1000
DPRIME_THRESHOLD = 0.3
OUT = drive.results("rui")
OUT.mkdir(parents=True, exist_ok=True)
{
    "areas": AREAS,
    "window_pairs": WINDOW_PAIRS,
    "neurons_per_area": NEURONS_PER_AREA,
    "dprime_threshold": DPRIME_THRESHOLD,
}


In [ ]:
manifest = db.query("""
    SELECT b.behavior_session_id, b.behavior_key, b.recording_id,
           b.experiment, b.mouse, b.cohort, e.stage, e.moment, b.trial_count
    FROM behavior_sessions AS b
    JOIN recordings AS r USING (recording_id)
    JOIN experiments AS e USING (experiment)
    WHERE r.has_behavior AND r.has_reduced_neural AND r.has_retinotopy
      AND e.stage = 'train1' AND e.moment IN ('before', 'after')
      AND b.cohort IN ('supervised', 'unsupervised')
    ORDER BY b.cohort, b.mouse, e.moment, b.recording_id
""")

In [ ]:
assert len(manifest) == 26
assert manifest["mouse"].nunique() == 13
assert set(manifest["moment"]) == {"before", "after"}
assert set(manifest["cohort"]) == {"supervised", "unsupervised"}


In [ ]:
probe = next(manifest.itertuples(index=False))
probe_result = session_dprime_history(
    db,
    probe,
    window_pairs=WINDOW_PAIRS,
    neurons_per_area=NEURONS_PER_AREA,
    threshold=DPRIME_THRESHOLD,
    include_all_areas=True,
    verify=True,
)
assert set(probe_result["area"]) == {"all", *AREAS}


In [ ]:
scan = [probe_result]
for index, session in enumerate(manifest.itertuples(index=False), start=1):
    if session.behavior_session_id == probe.behavior_session_id:
        continue
    started = time.perf_counter()
    result = session_dprime_history(
        db,
        session,
        window_pairs=WINDOW_PAIRS,
        neurons_per_area=NEURONS_PER_AREA,
        threshold=DPRIME_THRESHOLD,
        include_all_areas=True,
    )
    scan.append(result)
    print(f"{index:02d}/26 {session.behavior_session_id} {len(result):,} rows {time.perf_counter() - started:.1f}s")

history = pd.concat(scan, ignore_index=True)
history.shape


In [ ]:
history.to_csv(OUT / "window_dprime.csv", index=False)


In [ ]:
trajectory, independent, sessions, mouse_deltas, exact_tests = dprime_summary_tables(
    history,
    metrics=(
        "median", "sd_dprime", "frac_selective",
        "frac_leaf_selective", "frac_circle_selective",
    ),
    independent_window_pairs=WINDOW_PAIRS,
)
summary_outputs = {
    "mouse_first_trajectory.csv": trajectory,
    "nonoverlapping_gamm_windows.csv": independent,
    "session_dprime.csv": sessions,
    "mouse_deltas.csv": mouse_deltas,
    "exact_mouse_tests.csv": exact_tests,
}
for filename, frame in summary_outputs.items():
    frame.to_csv(OUT / filename, index=False)
{
    "descriptive_windows": history.shape,
    "nonoverlapping_area_windows": independent.shape,
    "session_area_rows": sessions.shape,
    "mouse_deltas": mouse_deltas.shape,
    "exact_tests": exact_tests.shape,
}


In [ ]:
import shutil

if importlib.util.find_spec("rpy2") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rpy2"], check=True)

if not IN_COLAB:
    os.environ.setdefault("RPY2_CFFI_MODE", "ABI")
    if "R_HOME" not in os.environ and shutil.which("R"):
        os.environ["R_HOME"] = subprocess.run(
            ["R", "RHOME"], capture_output=True, text=True, check=True
        ).stdout.strip()

get_ipython().run_line_magic("load_ext", "rpy2.ipython")


In [ ]:
%%R
suppressMessages(library(mgcv))

fit_pair_gamms <- function(pair_input) {
  df <- pair_input
  df$Mouse_ID <- factor(df$mouse)
  df$Window_ID <- df$window_id
  df$Running_Speed <- df$mean_run_speed
  df$Condition <- factor(
    ifelse(df$cohort == "supervised",
           ifelse(df$moment == "before", "sup_bef", "sup_aft"),
           ifelse(df$moment == "before", "unsup_bef", "unsup_aft")),
    levels = c("sup_bef", "sup_aft", "unsup_aft", "unsup_bef"))
  populations <- c("all", "V1", "mHV", "aHV", "lHV")
  responses <- c("median", "sd_dprime",
                 "frac_leaf_selective", "frac_circle_selective")
  pred_rows <- list()
  resid_rows <- list()
  diag_rows <- list()

  for (a in populations) {
    ad <- droplevels(subset(df, area == a))
    for (r in responses) {
      ad$response_value <- ad[[r]]
      m <- tryCatch(
        gam(response_value ~ Condition
              + s(Window_ID, by = Condition, bs = "cr", k = 5)
              + Running_Speed
              + s(Mouse_ID, bs = "re"),
            data = ad, method = "REML"),
        error = function(e) e)
      if (inherits(m, "error")) {
        diag_rows[[length(diag_rows) + 1]] <- data.frame(
          area = a, response = r, n = nrow(ad), mice = nlevels(ad$Mouse_ID),
          edf = NA, dev_expl = NA, r_sq = NA, reml = NA,
          converged = FALSE, message = conditionMessage(m),
          stringsAsFactors = FALSE)
        next
      }

      s <- summary(m)
      speed_reference <- mean(ad$Running_Speed, na.rm = TRUE)
      speed_beta <- as.numeric(coef(m)["Running_Speed"])
      resid_rows[[length(resid_rows) + 1]] <- data.frame(
        area = a, response = r, cohort = ad$cohort, moment = ad$moment,
        condition = as.character(ad$Condition), mouse = ad$mouse,
        window_id = ad$Window_ID, running_speed = ad$Running_Speed,
        observed = ad$response_value,
        adjusted = ad$response_value
          - speed_beta * (ad$Running_Speed - speed_reference),
        stringsAsFactors = FALSE)

      for (cond in levels(ad$Condition)) {
        observed_condition <- subset(ad, Condition == cond)
        grid <- data.frame(
          Condition = factor(cond, levels = levels(ad$Condition)),
          Window_ID = seq(min(observed_condition$Window_ID),
                          max(observed_condition$Window_ID), length.out = 80),
          Running_Speed = speed_reference,
          Mouse_ID = factor(levels(ad$Mouse_ID)[1],
                            levels = levels(ad$Mouse_ID)))
        pr <- predict(m, newdata = grid, se.fit = TRUE,
                      exclude = "s(Mouse_ID)")
        pred_rows[[length(pred_rows) + 1]] <- data.frame(
          area = a, response = r,
          cohort = ifelse(grepl("^sup_", cond),
                          "supervised", "unsupervised"),
          moment = ifelse(grepl("_bef$", cond), "before", "after"),
          condition = cond, window_id = grid$Window_ID,
          fit = as.numeric(pr$fit), se = as.numeric(pr$se.fit),
          stringsAsFactors = FALSE)
      }

      diag_rows[[length(diag_rows) + 1]] <- data.frame(
        area = a, response = r, n = nrow(ad), mice = nlevels(ad$Mouse_ID),
        edf = sum(m$edf), dev_expl = s$dev.expl, r_sq = s$r.sq,
        reml = as.numeric(m$gcv.ubre), converged = m$converged,
        message = "", stringsAsFactors = FALSE)
    }
  }

  list(
    predictions = do.call(rbind, pred_rows),
    partial_residuals = do.call(rbind, resid_rows),
    diagnostics = do.call(rbind, diag_rows))
}

In [ ]:
from functools import lru_cache

from rpy2 import robjects
from rpy2.robjects import conversion, default_converter, pandas2ri
from rpy2.robjects.conversion import localconverter

SUPERVISED_MICE = ["VR2", "TX60", "TX108", "TX109"]
UNSUPERVISED_MICE = [
    "TX83", "TX88", "TX105", "TX104", "TX119",
    "TX123", "DR10", "DR15", "TX85",
]
PAIR_POPULATIONS = ["all", "V1", "mHV", "aHV", "lHV"]
PAIR_POPULATION_LABELS = {
    "all": "All visual areas pooled",
    "V1": "V1",
    "mHV": "Medial higher visual area",
    "aHV": "Anterior higher visual area",
    "lHV": "Lateral higher visual area",
}
PAIR_RESPONSES = [
    "median", "sd_dprime", "frac_leaf_selective", "frac_circle_selective",
]
PAIR_RESPONSE_LABELS = {
    "median": "Median d′",
    "sd_dprime": "Standard deviation of d′",
    "frac_leaf_selective": "Fraction leaf-selective",
    "frac_circle_selective": "Fraction circle-selective",
}
PLOT_Y_LIMITS = {
    "median": (-0.40, 0.40),
    "sd_dprime": (0.10, 0.90),
    "frac_leaf_selective": (0.00, 0.60),
    "frac_circle_selective": (0.00, 0.60),
}
PAIR_COLORS = {"supervised": "#1b9e77", "unsupervised": "#d95f02"}
PAIR_PLOT_FILES = {}
PAIR_DIAGNOSTICS = {}
PAIR_OUT = OUT / "mouse_pair_plots"
PAIR_OUT.mkdir(parents=True, exist_ok=True)

@lru_cache(maxsize=None)
def fit_mouse_pair_gamms(supervised_mouse, unsupervised_mouse):
    if supervised_mouse not in SUPERVISED_MICE:
        raise ValueError(f"Unknown supervised mouse: {supervised_mouse}")
    if unsupervised_mouse not in UNSUPERVISED_MICE:
        raise ValueError(f"Unknown unsupervised mouse: {unsupervised_mouse}")

    pair_input = history.loc[
        history["mouse"].isin([supervised_mouse, unsupervised_mouse]),
        [
            "area", "cohort", "moment", "mouse", "window_id",
            "mean_run_speed", "median", "sd_dprime",
            "frac_leaf_selective", "frac_circle_selective",
        ],
    ].copy()
    observed = set(
        pair_input[["cohort", "moment"]]
        .drop_duplicates()
        .itertuples(index=False, name=None)
    )
    expected = {
        ("supervised", "before"), ("supervised", "after"),
        ("unsupervised", "before"), ("unsupervised", "after"),
    }
    assert observed == expected

    with localconverter(default_converter + pandas2ri.converter):
        r_input = conversion.py2rpy(pair_input)
    r_result = robjects.globalenv["fit_pair_gamms"](r_input)
    with localconverter(default_converter + pandas2ri.converter):
        predictions = conversion.rpy2py(r_result.rx2("predictions"))
        residuals = conversion.rpy2py(r_result.rx2("partial_residuals"))
        diagnostics = conversion.rpy2py(r_result.rx2("diagnostics"))

    diagnostics = diagnostics.assign(
        supervised_mouse=supervised_mouse,
        unsupervised_mouse=unsupervised_mouse,
    )
    if len(diagnostics) != len(PAIR_POPULATIONS) * len(PAIR_RESPONSES):
        raise RuntimeError("A pair did not return all 20 requested GAMMs")
    if not diagnostics["converged"].astype(bool).all():
        failed = diagnostics.loc[
            ~diagnostics["converged"].astype(bool),
            ["area", "response", "message"],
        ]
        raise RuntimeError(f"GAMM failure:\n{failed.to_string(index=False)}")

    PAIR_DIAGNOSTICS[(supervised_mouse, unsupervised_mouse)] = diagnostics
    return predictions, residuals, diagnostics

def plot_mouse_pair_moment(supervised_mouse, unsupervised_mouse, moment):
    if moment not in {"before", "after"}:
        raise ValueError("moment must be 'before' or 'after'")
    predictions, residuals, diagnostics = fit_mouse_pair_gamms(
        supervised_mouse, unsupervised_mouse
    )
    predictions = predictions.loc[predictions["moment"] == moment]
    residuals = residuals.loc[residuals["moment"] == moment]
    abbreviation = {"before": "bef", "after": "aft"}[moment]
    cohort_labels = {
        "supervised": f"Supervised ({supervised_mouse}, {abbreviation})",
        "unsupervised": f"Unsupervised ({unsupervised_mouse}, {abbreviation})",
    }

    fig, axes = plt.subplots(
        len(PAIR_POPULATIONS), len(PAIR_RESPONSES),
        figsize=(19, 18), sharex=True, sharey="col",
    )
    for row, population in enumerate(PAIR_POPULATIONS):
        for column, response in enumerate(PAIR_RESPONSES):
            axis = axes[row, column]
            dots = residuals.loc[
                (residuals["area"] == population)
                & (residuals["response"] == response)
            ]
            curves = predictions.loc[
                (predictions["area"] == population)
                & (predictions["response"] == response)
            ]
            for cohort, group in dots.groupby("cohort", sort=True):
                color = PAIR_COLORS[cohort]
                axis.scatter(
                    group["window_id"], group["adjusted"], s=12,
                    facecolor=color, edgecolor=color,
                    alpha=0.25, linewidth=0.55,
                )
            for cohort, group in curves.groupby("cohort", sort=True):
                color = PAIR_COLORS[cohort]
                axis.plot(
                    group["window_id"], group["fit"],
                    color=color, linewidth=2,
                    label=cohort_labels[cohort],
                )
                axis.fill_between(
                    group["window_id"],
                    group["fit"] - 1.96 * group["se"],
                    group["fit"] + 1.96 * group["se"],
                    color=color, alpha=0.20,
                )
            if row == 0:
                axis.set_title(PAIR_RESPONSE_LABELS[response], fontsize=12)
            if row == len(PAIR_POPULATIONS) - 1:
                axis.set_xlabel("Trial window index (session time)")
            axis.set_ylim(*PLOT_Y_LIMITS[response])
            axis.tick_params(labelsize=8)
            axis.spines[["top", "right"]].set_visible(False)

    fig.suptitle(
        f"Partial residual alignment — {moment.capitalize()} sessions\n"
        f"{supervised_mouse} (supervised) vs "
        f"{unsupervised_mouse} (unsupervised)",
        y=0.995, fontsize=18, fontweight="bold",
    )
    fig.text(
        0.5, 0.952,
        "Fixed y scale per metric; dots adjusted for speed; lines show GAMM fits",
        ha="center", va="top", fontsize=11,
    )
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(
        handles, labels, title="Condition", loc="upper center",
        bbox_to_anchor=(0.5, 0.925), ncol=2, frameon=False,
    )
    fig.tight_layout(rect=(0.06, 0.02, 1, 0.89), h_pad=2.0, w_pad=1.5)
    for row, population in enumerate(PAIR_POPULATIONS):
        position = axes[row, 0].get_position()
        fig.text(
            0.015, (position.y0 + position.y1) / 2,
            PAIR_POPULATION_LABELS[population],
            rotation=90, va="center", ha="center",
            fontsize=12, fontweight="bold",
        )

    filename = (
        f"gamm_{supervised_mouse}_vs_{unsupervised_mouse}_"
        f"{moment}_metrics.png"
    )
    fig.savefig(PAIR_OUT / filename, dpi=140, bbox_inches="tight")
    plt.show()

    record = {
        "supervised_mouse": supervised_mouse,
        "unsupervised_mouse": unsupervised_mouse,
        "moment": moment,
        "models": len(diagnostics),
        "all_converged": bool(diagnostics["converged"].astype(bool).all()),
        "file": str(Path("mouse_pair_plots") / filename),
    }
    PAIR_PLOT_FILES[(supervised_mouse, unsupervised_mouse, moment)] = record
    return pd.Series(record)

In [ ]:
plot_mouse_pair_moment("VR2", "TX83", "before")

In [ ]:
plot_mouse_pair_moment("VR2", "TX83", "after")

In [ ]:
plot_mouse_pair_moment("VR2", "TX88", "before")

In [ ]:
plot_mouse_pair_moment("VR2", "TX88", "after")

In [ ]:
plot_mouse_pair_moment("VR2", "TX105", "before")

In [ ]:
plot_mouse_pair_moment("VR2", "TX105", "after")

In [ ]:
plot_mouse_pair_moment("VR2", "TX104", "before")

In [ ]:
plot_mouse_pair_moment("VR2", "TX104", "after")

In [ ]:
plot_mouse_pair_moment("VR2", "TX119", "before")

In [ ]:
plot_mouse_pair_moment("VR2", "TX119", "after")

In [ ]:
plot_mouse_pair_moment("VR2", "TX123", "before")

In [ ]:
plot_mouse_pair_moment("VR2", "TX123", "after")

In [ ]:
plot_mouse_pair_moment("VR2", "DR10", "before")

In [ ]:
plot_mouse_pair_moment("VR2", "DR10", "after")

In [ ]:
plot_mouse_pair_moment("VR2", "DR15", "before")

In [ ]:
plot_mouse_pair_moment("VR2", "DR15", "after")

In [ ]:
plot_mouse_pair_moment("VR2", "TX85", "before")

In [ ]:
plot_mouse_pair_moment("VR2", "TX85", "after")

In [ ]:
plot_mouse_pair_moment("TX60", "TX83", "before")

In [ ]:
plot_mouse_pair_moment("TX60", "TX83", "after")

In [ ]:
plot_mouse_pair_moment("TX60", "TX88", "before")

In [ ]:
plot_mouse_pair_moment("TX60", "TX88", "after")

In [ ]:
plot_mouse_pair_moment("TX60", "TX105", "before")

In [ ]:
plot_mouse_pair_moment("TX60", "TX105", "after")

In [ ]:
plot_mouse_pair_moment("TX60", "TX104", "before")

In [ ]:
plot_mouse_pair_moment("TX60", "TX104", "after")

In [ ]:
plot_mouse_pair_moment("TX60", "TX119", "before")

In [ ]:
plot_mouse_pair_moment("TX60", "TX119", "after")

In [ ]:
plot_mouse_pair_moment("TX60", "TX123", "before")

In [ ]:
plot_mouse_pair_moment("TX60", "TX123", "after")

In [ ]:
plot_mouse_pair_moment("TX60", "DR10", "before")

In [ ]:
plot_mouse_pair_moment("TX60", "DR10", "after")

In [ ]:
plot_mouse_pair_moment("TX60", "DR15", "before")

In [ ]:
plot_mouse_pair_moment("TX60", "DR15", "after")

In [ ]:
plot_mouse_pair_moment("TX60", "TX85", "before")

In [ ]:
plot_mouse_pair_moment("TX60", "TX85", "after")

In [ ]:
plot_mouse_pair_moment("TX108", "TX83", "before")

In [ ]:
plot_mouse_pair_moment("TX108", "TX83", "after")

In [ ]:
plot_mouse_pair_moment("TX108", "TX88", "before")

In [ ]:
plot_mouse_pair_moment("TX108", "TX88", "after")

In [ ]:
plot_mouse_pair_moment("TX108", "TX105", "before")

In [ ]:
plot_mouse_pair_moment("TX108", "TX105", "after")

In [ ]:
plot_mouse_pair_moment("TX108", "TX104", "before")

In [ ]:
plot_mouse_pair_moment("TX108", "TX104", "after")

In [ ]:
plot_mouse_pair_moment("TX108", "TX119", "before")

In [ ]:
plot_mouse_pair_moment("TX108", "TX119", "after")

In [ ]:
plot_mouse_pair_moment("TX108", "TX123", "before")

In [ ]:
plot_mouse_pair_moment("TX108", "TX123", "after")

In [ ]:
plot_mouse_pair_moment("TX108", "DR10", "before")

In [ ]:
plot_mouse_pair_moment("TX108", "DR10", "after")

In [ ]:
plot_mouse_pair_moment("TX108", "DR15", "before")

In [ ]:
plot_mouse_pair_moment("TX108", "DR15", "after")

In [ ]:
plot_mouse_pair_moment("TX108", "TX85", "before")

In [ ]:
plot_mouse_pair_moment("TX108", "TX85", "after")

In [ ]:
plot_mouse_pair_moment("TX109", "TX83", "before")

In [ ]:
plot_mouse_pair_moment("TX109", "TX83", "after")

In [ ]:
plot_mouse_pair_moment("TX109", "TX88", "before")

In [ ]:
plot_mouse_pair_moment("TX109", "TX88", "after")

In [ ]:
plot_mouse_pair_moment("TX109", "TX105", "before")

In [ ]:
plot_mouse_pair_moment("TX109", "TX105", "after")

In [ ]:
plot_mouse_pair_moment("TX109", "TX104", "before")

In [ ]:
plot_mouse_pair_moment("TX109", "TX104", "after")

In [ ]:
plot_mouse_pair_moment("TX109", "TX119", "before")

In [ ]:
plot_mouse_pair_moment("TX109", "TX119", "after")

In [ ]:
plot_mouse_pair_moment("TX109", "TX123", "before")

In [ ]:
plot_mouse_pair_moment("TX109", "TX123", "after")

In [ ]:
plot_mouse_pair_moment("TX109", "DR10", "before")

In [ ]:
plot_mouse_pair_moment("TX109", "DR10", "after")

In [ ]:
plot_mouse_pair_moment("TX109", "DR15", "before")

In [ ]:
plot_mouse_pair_moment("TX109", "DR15", "after")

In [ ]:
plot_mouse_pair_moment("TX109", "TX85", "before")

In [ ]:
plot_mouse_pair_moment("TX109", "TX85", "after")

In [ ]:
pair_plot_manifest = pd.DataFrame(PAIR_PLOT_FILES.values()).sort_values(
    ["supervised_mouse", "unsupervised_mouse", "moment"]
)
pair_gamm_diagnostics = pd.concat(
    PAIR_DIAGNOSTICS.values(), ignore_index=True
).sort_values(["supervised_mouse", "unsupervised_mouse", "area", "response"])
pair_plot_manifest.to_csv(OUT / "mouse_pair_plot_files.csv", index=False)
pair_gamm_diagnostics.to_csv(
    OUT / "mouse_pair_gamm_diagnostics.csv", index=False
)
{
    "mouse_pairs": len(PAIR_DIAGNOSTICS),
    "moment_plots": len(pair_plot_manifest),
    "models": len(pair_gamm_diagnostics),
    "all_converged": bool(pair_gamm_diagnostics["converged"].astype(bool).all()),
}